In [31]:
# imports
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

import tensorflow as tf
from keras.layers import TextVectorization
from keras.models import load_model

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('punkt')

# initialise random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/davidf_wsl/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/davidf_wsl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [32]:
DATA_DIR = Path("../Kaggle_FakeAndRealNewsDataset")
FAKE_NEWS_CSV = DATA_DIR / "Fake.csv"
TRUE_NEWS_CSV = DATA_DIR / "True.csv"

# validate dataset path
def validate_csv_path(path, label):
    if not path.exists():
        raise FileNotFoundError(f">> [ERROR] {label} not found at: {path.resolve()}\n"
                                f">> [INFO] Download from: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
    if not path.is_file():
        raise ValueError(f">> [ERROR] {label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f">> [ERROR] {label} is not a CSV file: {path.resolve()}")
    
    print(f">> [OK] {label}: {path.resolve()}")
    
validate_csv_path(FAKE_NEWS_CSV, "Fake news dataset")
validate_csv_path(TRUE_NEWS_CSV, "True news dataset")

>> [OK] Fake news dataset: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Kaggle_FakeAndRealNewsDataset/Fake.csv
>> [OK] True news dataset: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Kaggle_FakeAndRealNewsDataset/True.csv


In [33]:
df_false = pd.read_csv(FAKE_NEWS_CSV)
df_true = pd.read_csv(TRUE_NEWS_CSV)

# Combine datasets
df = pd.concat([df_false, df_true], ignore_index=True)

# Drop rows with missing text or subject
df = df.dropna(subset=["text", "subject"])
df = df[df["text"].str.strip() != ""]

# Combine pliticalNews with politics and worldnews with news to achieve the reported 6 cathegories
df['subject'] = df['subject'].replace({
    'politicsNews': 'politics', 
    'worldnews': 'News'
    }) 

print(f">> Dataset Shape: {df.shape}")
print(f"\n>> --- [Subject Distribution] --- :\n")

display(pd.DataFrame(df['subject'].value_counts()). rename(columns={'subject': 'count'}).reset_index())

>> Dataset Shape: (44267, 4)

>> --- [Subject Distribution] --- :



,subject,count
0,News,19195
1,politics,17704
2,left-news,4309
3,Government News,1498
4,US_News,783
5,Middle-east,778


In [34]:
# get stop words
stop_words = set(stopwords.words("english"))

# def function to preprocess text - lowercase, remove punctuation, remove stop words
def preprocess(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    
    # join tokens back to string
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)

print(">> Preprocessing complete")
display(df.head())

>> Preprocessing complete


,title,text,subject,date,clean_text
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",donald trump wish americans happy new year lea...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",house intelligence committee chairman devin nu...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",friday revealed former milwaukee sheriff david...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",christmas day donald trump announced would bac...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",pope francis used annual christmas day message...


In [35]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["subject"])

print(f">> --- [Classes] --- :\n{label_encoder.classes_}")
print("\n>> Label mapping:")

for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")
 
NUM_CLASSES = len(label_encoder.classes_)
print(f"\n>> Number of classes: {NUM_CLASSES}")

>> --- [Classes] --- :
['Government News' 'Middle-east' 'News' 'US_News' 'left-news' 'politics']

>> Label mapping:
  0: Government News
  1: Middle-east
  2: News
  3: US_News
  4: left-news
  5: politics

>> Number of classes: 6


In [36]:
X_text = df["clean_text"].values
y = df["label"].values

# (Paper-aligned) 70-30 split for training and testing, stratified by label
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

print(f">> Train set: {len(X_train_text)} samples ({len(X_train_text)/len(X_text)*100:.2f}%)")
print(f">> Test set:  {len(X_test_text)} samples ({len(X_test_text)/len(X_text)*100:.2f}%)")

>> Train set: 30986 samples (70.00%)
>> Test set:  13281 samples (30.00%)


In [37]:
VOCAB_SIZE  = [10_000, 101_637, 200_000]    # 101,637
SEQ_LENGTH  = 500        # Sequence Length 500
EMBED_DIM   = 50         # Embedding Dimension 50

seq_test_by_vocab = {}

for vocab_size in VOCAB_SIZE:
    # Vectorise text on the full training set to build the vocabulary
    vectorise_layer = TextVectorization(
        max_tokens=vocab_size,       # vocabulary cap
        output_mode="int",           # integer indices (same as old texts_to_sequences)
        output_sequence_length=SEQ_LENGTH,  # pads/truncates to fixed length (replaces pad_sequences)
        name=f"text_vectorization_{vocab_size}"  # unique name per vocab size
    )
    vectorise_layer.adapt(X_train_text)
    
    X_seq_test = vectorise_layer(X_test_text).numpy()
    seq_test_by_vocab[vocab_size] = X_seq_test

In [38]:
wORn_es = "With"  # With or No
wORn_cw = "With"  # With or No
MODEL_DIR = Path(f"../Out/Models_{wORn_es}EarlyStopping_{wORn_cw}ClassWeights")

# If needed, you can change MODEL_DIR manually here.
print("Model directory:", MODEL_DIR.resolve())
print("Exists:", MODEL_DIR.exists())

expected_models = {
    "CNN-LSTM": {
        10_000: MODEL_DIR / "cnn_lstm_model_vocab10000.keras",
        101_637: MODEL_DIR / "cnn_lstm_model_vocab101637.keras",
        200_000: MODEL_DIR / "cnn_lstm_model_vocab200000.keras",
    },
    "CNN-LSTM + Modifications": {
        10_000: MODEL_DIR / "cnn_lstm_modified_model_vocab10000.keras",
        101_637: MODEL_DIR / "cnn_lstm_modified_model_vocab101637.keras",
        200_000: MODEL_DIR / "cnn_lstm_modified_model_vocab200000.keras",
    },
}

for family, family_models in expected_models.items():
    print(f"\n{family}")
    for vocab_size, model_path in family_models.items():
        print(f"  Vocab {vocab_size:,}: {'FOUND' if model_path.exists() else 'MISSING'} -> {model_path.name}")

Model directory: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_WithEarlyStopping_WithClassWeights
Exists: True

CNN-LSTM
  Vocab 10,000: FOUND -> cnn_lstm_model_vocab10000.keras
  Vocab 101,637: FOUND -> cnn_lstm_model_vocab101637.keras
  Vocab 200,000: FOUND -> cnn_lstm_model_vocab200000.keras

CNN-LSTM + Modifications
  Vocab 10,000: FOUND -> cnn_lstm_modified_model_vocab10000.keras
  Vocab 101,637: FOUND -> cnn_lstm_modified_model_vocab101637.keras
  Vocab 200,000: FOUND -> cnn_lstm_modified_model_vocab200000.keras


In [39]:
results = []
reports = {}

for family, family_models in expected_models.items():
    for vocab_size, model_path in family_models.items():
        if not model_path.exists():
            print(f"Skipping missing model: {model_path.name}")
            continue

        print(f"Evaluating: {model_path.name}")

        model = load_model(model_path, compile=False)
        X_seq_test = seq_test_by_vocab[vocab_size]

        y_pred_probs = model.predict(X_seq_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)

        acc = accuracy_score(y_test, y_pred)
        weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        results.append({
            "model_family": family,
            "vocab_size": vocab_size,
            "file_name": model_path.name,
            "accuracy": acc,
            "weighted_f1": weighted_f1
        })

        reports[f"{family} | vocab {vocab_size:,}"] = classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )

results_df = pd.DataFrame(results).sort_values(
    by=["weighted_f1", "accuracy"],
    ascending=False
).reset_index(drop=True)

display(results_df)

if results_df.empty:
    print("No saved models were found in the selected model directory.")


Evaluating: cnn_lstm_model_vocab10000.keras
Evaluating: cnn_lstm_model_vocab101637.keras
Evaluating: cnn_lstm_model_vocab200000.keras
Evaluating: cnn_lstm_modified_model_vocab10000.keras
Evaluating: cnn_lstm_modified_model_vocab101637.keras
Evaluating: cnn_lstm_modified_model_vocab200000.keras


,model_family,vocab_size,file_name,accuracy,weighted_f1
0,CNN-LSTM,200000,cnn_lstm_model_vocab200000.keras,0.742640,0.768777
1,CNN-LSTM + Modifications,101637,cnn_lstm_modified_model_vocab101637.keras,0.740230,0.765726
2,CNN-LSTM + Modifications,200000,cnn_lstm_modified_model_vocab200000.keras,0.749191,0.763388
3,CNN-LSTM + Modifications,10000,cnn_lstm_modified_model_vocab10000.keras,0.736993,0.752821
4,CNN-LSTM,10000,cnn_lstm_model_vocab10000.keras,0.747007,0.751538
5,CNN-LSTM,101637,cnn_lstm_model_vocab101637.keras,0.734734,0.749140


In [40]:
# Optional: show rounded summary only
if not results_df.empty:
    summary_df = results_df.copy()
    summary_df["accuracy"] = summary_df["accuracy"].round(3)
    summary_df["weighted_f1"] = summary_df["weighted_f1"].round(3)
    display(summary_df[["model_family", "vocab_size", "accuracy", "weighted_f1", "file_name"]])


,model_family,vocab_size,accuracy,weighted_f1,file_name
0,CNN-LSTM,200000,0.743,0.769,cnn_lstm_model_vocab200000.keras
1,CNN-LSTM + Modifications,101637,0.740,0.766,cnn_lstm_modified_model_vocab101637.keras
2,CNN-LSTM + Modifications,200000,0.749,0.763,cnn_lstm_modified_model_vocab200000.keras
3,CNN-LSTM + Modifications,10000,0.737,0.753,cnn_lstm_modified_model_vocab10000.keras
4,CNN-LSTM,10000,0.747,0.752,cnn_lstm_model_vocab10000.keras
5,CNN-LSTM,101637,0.735,0.749,cnn_lstm_model_vocab101637.keras
